# Issam — Batch Test All 100 Bundled Samples

Runs the issam model on **every** clip in `sample_videos/by_class/` (SignID **71–170**).

Use this to confirm each bundled video maps to the **correct class label**.

**Run:** Cell 1 → 2 → 3 → 4

See `sample_videos/manifest.csv` for SignID ↔ filename ↔ Arabic/English names.

In [ ]:
# CELL 1 — imports
import os
import csv
import cv2
import numpy as np
import pandas as pd
import mediapipe as mp
import tensorflow as tf
from pathlib import Path

os.environ.setdefault('PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION', 'python')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')

print(f'TF {tf.__version__}  |  MP {mp.__version__}')

In [ ]:
# CELL 2 — bundle paths + model + manifest
from bundle_config import find_bundle_from_cwd, MODEL_PATH, BUNDLE_SAMPLES, MANIFEST_CSV, RESULTS_CSV
from bundle_config import F_AVG, N_FEATURES

DIR = find_bundle_from_cwd()
LABELS_XLSX = DIR / 'KARSL-502_Labels.xlsx'
print(f'Bundle root: {DIR}')
karsl_df = pd.read_excel(DIR / 'KARSL-502_Labels.xlsx').iloc[70:170].reset_index(drop=True)
words_en = karsl_df['Sign-English'].astype(str).str.strip().to_numpy()
words_ar = karsl_df['Sign-Arabic'].astype(str).str.strip().to_numpy()
sign_ids = karsl_df['SignID'].astype(int).to_numpy()
num_classes = len(sign_ids)
sid_to_idx = {int(s): i for i, s in enumerate(sign_ids)}

def build_model(n_classes):
    return tf.keras.Sequential([
        tf.keras.layers.Input(shape=(F_AVG, N_FEATURES)),
        tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, return_sequences=True)),
        tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64)),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dense(n_classes, activation='softmax'),
    ])

try:
    model = tf.keras.models.load_model(MODEL_PATH, compile=False)
except Exception:
    model = build_model(num_classes)
    model.load_weights(MODEL_PATH)

if MANIFEST_CSV.exists():
    manifest = pd.read_csv(MANIFEST_CSV)
else:
    manifest = pd.DataFrame({
        'class_index': range(num_classes),
        'sign_id': sign_ids,
        'english': words_en,
        'arabic': words_ar,
        'filename': [f'SignID{int(s):04d}.mp4' for s in sign_ids],
    })

on_disk = sum(1 for s in sign_ids if (BUNDLE_SAMPLES / f'SignID{int(s):04d}.mp4').is_file())
print(f'Bundle : {DIR}')
print(f'Samples: {on_disk}/{num_classes} in {BUNDLE_SAMPLES}')
assert on_disk == num_classes, f'Missing samples — run: python scripts/copy_class_samples.py'

In [ ]:
# CELL 3 — issam 225-D extraction
mp_holistic = mp.solutions.holistic
holistic = mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5)

def mediapipe_detection(bgr):
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    rgb.flags.writeable = False
    return holistic.process(rgb)

def adjust_landmarks(arr, center):
    arr_reshaped = arr.reshape(-1, 3)
    return (arr_reshaped - np.tile(center, (len(arr_reshaped), 1))).reshape(-1)

def extract_keypoints(results):
    pose = np.array([[r.x, r.y, r.z] for r in results.pose_landmarks.landmark]).flatten() if results.pose_landmarks else np.zeros(99)
    lh = np.array([[r.x, r.y, r.z] for r in results.left_hand_landmarks.landmark]).flatten() if results.left_hand_landmarks else np.zeros(63)
    rh = np.array([[r.x, r.y, r.z] for r in results.right_hand_landmarks.landmark]).flatten() if results.right_hand_landmarks else np.zeros(63)
    return np.concatenate([
        adjust_landmarks(pose, pose[:3]),
        adjust_landmarks(lh, lh[:3]),
        adjust_landmarks(rh, rh[:3]),
    ]).astype(np.float32)

def pad_or_trim(seq, f_avg=F_AVG):
    seq = np.asarray(seq, dtype=np.float32)
    n = min(seq.shape[0], f_avg)
    seq = seq[:n, :]
    while seq.shape[0] < f_avg:
        seq = np.concatenate([seq, seq[-1:, :]], axis=0)
    return seq

def predict_video_path(video_path):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise FileNotFoundError(video_path)
    seq = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        seq.append(extract_keypoints(mediapipe_detection(frame)))
    cap.release()
    probs = model.predict(pad_or_trim(seq)[None, ...], verbose=0)[0]
    top = int(np.argmax(probs))
    return top, float(probs[top]), len(seq)

In [ ]:
# CELL 4 — batch run + summary
rows = []
correct = 0

for i in range(num_classes):
    sid = int(sign_ids[i])
    vid = BUNDLE_SAMPLES / f'SignID{sid:04d}.mp4'
    pred_i, conf, n_frames = predict_video_path(vid)
    ok = pred_i == i
    correct += int(ok)
    rows.append({
        'class_index': i,
        'sign_id': sid,
        'expected_en': words_en[i],
        'expected_ar': words_ar[i],
        'filename': vid.name,
        'predicted_index': pred_i,
        'predicted_sign_id': int(sign_ids[pred_i]),
        'predicted_en': words_en[pred_i],
        'confidence': round(conf, 4),
        'frames': n_frames,
        'correct': ok,
    })
    mark = 'OK' if ok else 'WRONG'
    print(f'[{i+1:3d}/100] SignID {sid}  {mark:5s}  {conf:5.1%}  exp={words_en[i][:20]:20s}  got={words_en[pred_i][:20]}')

holistic.close()
results_df = pd.DataFrame(rows)
results_df.to_csv(RESULTS_CSV, index=False, encoding='utf-8-sig')

acc = correct / num_classes
wrong = results_df[~results_df['correct']]
print('\n' + '=' * 60)
print(f'Top-1 accuracy on bundled samples: {correct}/{num_classes} = {acc:.1%}')
print(f'Results saved: {RESULTS_CSV.name}')
if len(wrong):
    print(f'\nMisclassified ({len(wrong)}):')
    display(wrong[['sign_id', 'expected_en', 'predicted_en', 'confidence']])
else:
    print('All 100 bundled samples classified correctly.')